# Análisis Exploratorio de Datos (EDA) - Airbnb Mexico


## Objetivo
Entender la estructura, calidad y características de los datos antes de realizar transformaciones.

**Colecciones a analizar:**
- `Listings`: Información detallada de los apartamentos
- `Reviews`: Información de reseñas de clientes
- `Calendar`: Información de disponibilidad y precios

## Importaciones

In [1]:
import sys
from pathlib import Path

# Raíz del proyecto: sirve si el cwd es `notebooks/` o la raíz del repo
_cwd = Path.cwd().resolve()
if (_cwd / "src").is_dir():
    _root = _cwd
elif (_cwd.parent / "src").is_dir():
    _root = _cwd.parent
else:
    _root = _cwd.parent
sys.path.insert(0, str(_root))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import Extraccion, get_logger

# Configuración de visualizaciones
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 5)

logger = get_logger(__name__)
print("✓ Importaciones completadas")

✓ Importaciones completadas


## Funciones comunes

In [2]:
from IPython.display import display

def analizar_estructura(df, nombre_coleccion):
    """
    Realiza análisis general de estructura de un DataFrame.
    
    Muestra:
    - Primeras filas
    - Dimensiones
    - Tipos de datos
    - Estadísticas numéricas
    - Estadísticas generales
    - Resumen de variables no numéricas
    """
    print("\n" + "=" * 80)
    print(f"COLECCIÓN: {nombre_coleccion}")
    print("=" * 80)
    
    print(f"\nPRIMERAS FILAS (primeros 5 registros):")
    display(df.head(5))
    
    print(f"\nDIMENSIONES:")
    print(f"   - Filas: {df.shape[0]:,}")
    print(f"   - Columnas: {df.shape[1]}")
    
    print(f"\nTIPOS DE DATOS:")
    print("INFO")
    df.info(verbose=False)
    print("TYPES")
    print(df.dtypes)
    
    print(f"\nESTADÍSTICAS (Columnas numéricas):")
    display(df.describe().T)
    
    print(f"\nESTADÍSTICAS COMPLETAS (incluye variables categóricas):")
    display(df.describe(include='all').T)
    
    cols_no_numericas = df.select_dtypes(include=['object', 'string', 'bool', 'datetime64', 'str']).columns
    if len(cols_no_numericas) > 0:
        print(f"\nCOLUMNAS NO NUMÉRICAS ({len(cols_no_numericas)}):")
        for col in cols_no_numericas:
            try:
                unicos = df[col].nunique(dropna=False)
                print(f"   - {col}: {unicos} valores únicos")
                top = df[col].value_counts(dropna=False).head(5)
                print(f"      Principales valores:\n{top.to_dict()}")
            except Exception:
                print(f"   - {col}: Contiene datos complejos o no se puede resumir")


def analizar_valores_nulos(df, nombre_coleccion):
    """
    Analiza valores nulos y faltantes en un DataFrame.
    
    Muestra:
    - Cantidad de nulos por columna
    - Porcentaje respecto al total de registros
    - Solo columnas que tienen al menos un nulo
    """
    print(f"\nVALORES NULOS Y FALTANTES - {nombre_coleccion}:")
    print("-" * 80)
    
    nulos = df.isnull().sum()
    total_registros = len(df)
    nulos_pct = (nulos / total_registros * 100).round(2)
    
    df_nulos = pd.DataFrame({
        'Columna': nulos.index,
        'Cantidad de nulos': nulos.values,
        'Porcentaje (%)': nulos_pct.values
    })
    
    # Filtrar solo columnas con nulos
    df_nulos_filtrado = df_nulos[df_nulos['Cantidad de nulos'] > 0].sort_values(
        'Cantidad de nulos', ascending=False
    )
    
    if len(df_nulos_filtrado) > 0:
        print(f"   Columnas con valores faltantes:")
        display(df_nulos_filtrado)
    else:
        print(f"   Todas las columnas están completas (sin nulos)")


def analizar_duplicados(df, nombre_coleccion, col_id='_id'):
    """
    Analiza registros duplicados de dos formas:
    1. Duplicados por columna ID (identidad única)
    2. Filas completamente duplicadas en todas sus columnas
    """
    print(f"\nREGISTROS DUPLICADOS - {nombre_coleccion}:")
    print("-" * 80)
    
    total = len(df)
    
    # Análisis 1: Duplicados por ID
    unicos = df[col_id].nunique()
    duplicados_id = total - unicos
    pct_duplicados_id = (duplicados_id / total * 100) if total > 0 else 0
    
    print(f"   1. POR COLUMNA ID ({col_id}):")
    print(f"      - Total de registros: {total:,}")
    print(f"      - Registros únicos: {unicos:,}")
    print(f"      - Duplicados por ID: {duplicados_id:,} ({pct_duplicados_id:.2f}%)")
    
    # Análisis 2: Filas completamente duplicadas
    filas_duplicadas = df.duplicated().sum()
    pct_filas_dup = (filas_duplicadas / total * 100) if total > 0 else 0
    
    print(f"\n   2. FILAS COMPLETAMENTE DUPLICADAS (todas las columnas):")
    print(f"      - Filas duplicadas: {filas_duplicadas:,} ({pct_filas_dup:.2f}%)")
    
    if duplicados_id == 0 and filas_duplicadas == 0:
        print(f"\n   Excelente: No hay duplicados en esta colección")


def analizar_outliers(df, nombre_coleccion, columnas=None):
    """
    Analiza outliers en columnas numéricas usando método de cuartiles (IQR).
    """
    if columnas is None:
        columnas = df.select_dtypes(include='number').columns.tolist()
    
    print(f"\nANÁLISIS DE OUTLIERS - {nombre_coleccion}:")
    print("-" * 80)
    
    if len(columnas) == 0:
        print("   No hay columnas numéricas para analizar")
        return
    
    for col in columnas:
        if col not in df.columns:
            print(f"   Columna '{col}' no encontrada")
            continue
            
        datos = pd.to_numeric(df[col], errors='coerce').dropna()
        
        if len(datos) == 0:
            print(f"\n   {col}: No hay datos numéricos válidos")
            continue
        
        Q1 = datos.quantile(0.25)
        Q3 = datos.quantile(0.75)
        IQR = Q3 - Q1
        
        if IQR == 0:
            print(f"\n   {col}: IQR = 0, no hay variación suficiente para detectar outliers con este método")
            continue
        
        limite_inferior = Q1 - 1.5 * IQR
        limite_superior = Q3 + 1.5 * IQR
        
        outliers = datos[(datos < limite_inferior) | (datos > limite_superior)]
        pct_outliers = len(outliers) / len(datos) * 100
        
        print(f"\n   {col}:")
        print(f"      - Mínimo: {datos.min():.2f}")
        print(f"      - Q1 (25%): {Q1:.2f}")
        print(f"      - Mediana: {datos.median():.2f}")
        print(f"      - Q3 (75%): {Q3:.2f}")
        print(f"      - Máximo: {datos.max():.2f}")
        print(f"      - IQR: {IQR:.2f}")
        print(f"      - Outliers detectados: {len(outliers):,} ({pct_outliers:.2f}%)")
        print(f"      - Rango válido: [{limite_inferior:.2f}, {limite_superior:.2f}]")


## Extracción de datos desde MongoDB

In [3]:
# Conectar a MongoDB y extraer datos
extraccion = Extraccion()
print("Conectando a MongoDB...")
extraccion.conectar()

print("\nExtrayendo colecciones...")
df_listings = extraccion.extraer_coleccion("Listings")
df_reviews = extraccion.extraer_coleccion("Reviews")
df_calendar = extraccion.extraer_coleccion("Calendar")

print("\nExtracción completada")
print(f"\nRESUMEN:")
print(f"   - Listings: {len(df_listings):,} registros, {len(df_listings.columns)} columnas")
print(f"   - Reviews: {len(df_reviews):,} registros, {len(df_reviews.columns)} columnas")
print(f"   - Calendar: {len(df_calendar):,} registros, {len(df_calendar.columns)} columnas")

2026-04-12 17:00:52,674 | INFO | Extraccion | Conexion exitosa a MongoDB. Base de datos seleccionada: arbnb_MXN


Conectando a MongoDB...

Extrayendo colecciones...


2026-04-12 17:00:52,873 | INFO | Extraccion | Coleccion Listings extraida correctamente. Registros encontrados: 27051. Registros cargados en DataFrame: 27051.
2026-04-12 17:00:57,708 | INFO | Extraccion | Coleccion Reviews extraida correctamente. Registros encontrados: 1454740. Registros cargados en DataFrame: 1454740.
2026-04-12 17:01:41,878 | INFO | Extraccion | Coleccion Calendar extraida correctamente. Registros encontrados: 9873624. Registros cargados en DataFrame: 9873624.



Extracción completada

RESUMEN:
   - Listings: 27,051 registros, 17 columnas
   - Reviews: 1,454,740 registros, 3 columnas
   - Calendar: 9,873,624 registros, 6 columnas


## 1. Análisis de Colección: LISTINGS

Información detallada sobre los apartamentos disponibles en Airbnb Buenos Aires.

### Análisis de estructura

In [4]:
analizar_estructura(df_listings, "LISTINGS")


COLECCIÓN: LISTINGS

PRIMERAS FILAS (primeros 5 registros):


,_id,id,name,host_id,host_name,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,calculated_host_listings_count,availability_365,number_of_reviews_ltm,last_review,reviews_per_month
0,69dbd58adcaeaf06fe6ef9f8,35797,Villa Dante,153786,Dici,Cuajimalpa de Morelos,19.38283,-99.27178,Entire home/apt,3673.0,1,0,1,363,0,NaT,NaN
1,69dbd58adcaeaf06fe6ef9f9,44616,Condesa Haus,196253,Fernando,Cuauhtémoc,19.41162,-99.17794,Entire home/apt,18000.0,1,65,9,360,1,2025-01-01,0.38
2,69dbd58adcaeaf06fe6ef9fa,56074,Great space in historical San Rafael,265650,Maris,Cuauhtémoc,19.43977,-99.15605,Entire home/apt,591.0,15,84,1,333,1,2025-02-27,0.48
3,69dbd58adcaeaf06fe6ef9fb,67703,"2 bedroom apt. deco bldg, Condesa",334451,Nicholas,Cuauhtémoc,19.41152,-99.16857,Entire home/apt,NaN,2,50,2,252,1,2024-10-30,0.30
4,69dbd58adcaeaf06fe6ef9fc,70644,Beautiful light Studio Coyoacan- full equipped !,212109,Trisha,Coyoacán,19.35448,-99.16217,Entire home/apt,NaN,3,134,3,234,8,2025-08-18,0.81



DIMENSIONES:
   - Filas: 27,051
   - Columnas: 17

TIPOS DE DATOS:
INFO
<class 'pandas.DataFrame'>
RangeIndex: 27051 entries, 0 to 27050
Columns: 17 entries, _id to reviews_per_month
dtypes: datetime64[us](1), float64(4), int64(7), object(2), str(3)
memory usage: 3.5+ MB
TYPES
_id                                          str
id                                         int64
name                                      object
host_id                                    int64
host_name                                 object
neighbourhood                                str
latitude                                 float64
longitude                                float64
room_type                                    str
price                                    float64
minimum_nights                             int64
number_of_reviews                          int64
calculated_host_listings_count             int64
availability_365                           int64
number_of_reviews_ltm              

,count,mean,min,25%,50%,75%,max,std
id,27051.0,700355582609886464.0,35797.0,44104725.0,830736761677326976.0,1217382011952653056.0,1518561444972990720.0,570271415889610304.0
host_id,27051.0,242943885.020554,7365.0,56214432.0,176683531.0,428882972.0,720844507.0,206257879.87984
latitude,27051.0,19.40549,19.177848,19.392489,19.41539,19.432015,19.56101,0.04239
longitude,27051.0,-99.165385,-99.33963,-99.178499,-99.1671,-99.153731,-98.96336,0.033535
price,23567.0,1792.540841,61.0,643.0,1039.0,1611.0,900000.0,13230.940558
minimum_nights,27051.0,4.580866,1.0,1.0,2.0,2.0,1125.0,24.784605
number_of_reviews,27051.0,53.777679,0.0,4.0,21.0,69.0,1434.0,85.143043
calculated_host_listings_count,27051.0,14.559868,1.0,1.0,3.0,11.0,221.0,31.892178
availability_365,27051.0,232.686333,0.0,140.0,269.0,341.0,365.0,122.270858
number_of_reviews_ltm,27051.0,15.691028,0.0,0.0,7.0,23.0,665.0,23.615517



ESTADÍSTICAS COMPLETAS (incluye variables categóricas):


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
_id,27051,27051,69dbd58adcaeaf06fe6ef9f8,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
id,27051.0,NaN,NaN,NaN,700355582609886464.0,35797.0,44104725.0,830736761677326976.0,1217382011952653056.0,1518561444972990720.0,570271415889610304.0
name,27051,25638,"Blueground | Roma Sur 1 recamara, AC & rooftop",52,NaN,NaN,NaN,NaN,NaN,NaN,NaN
host_id,27051.0,NaN,NaN,NaN,242943885.020554,7365.0,56214432.0,176683531.0,428882972.0,720844507.0,206257879.87984
host_name,27045,4299,Alejandro,388,NaN,NaN,NaN,NaN,NaN,NaN,NaN
neighbourhood,27051,16,Cuauhtémoc,12514,NaN,NaN,NaN,NaN,NaN,NaN,NaN
latitude,27051.0,NaN,NaN,NaN,19.40549,19.177848,19.392489,19.41539,19.432015,19.56101,0.04239
longitude,27051.0,NaN,NaN,NaN,-99.165385,-99.33963,-99.178499,-99.1671,-99.153731,-98.96336,0.033535
room_type,27051,4,Entire home/apt,17713,NaN,NaN,NaN,NaN,NaN,NaN,NaN
price,23567.0,NaN,NaN,NaN,1792.540841,61.0,643.0,1039.0,1611.0,900000.0,13230.940558



COLUMNAS NO NUMÉRICAS (6):
   - _id: 27051 valores únicos
      Principales valores:
{'69dbd58adcaeaf06fe6ef9f8': 1, '69dbd58adcaeaf06fe6ef9f9': 1, '69dbd58adcaeaf06fe6ef9fa': 1, '69dbd58adcaeaf06fe6ef9fb': 1, '69dbd58adcaeaf06fe6ef9fc': 1}
   - name: 25638 valores únicos
      Principales valores:
{'Blueground | Roma Sur 1 recamara, AC & rooftop': 52, 'Perfecto Loft en gran ubicación': 48, 'Blueground | Polanco, Luxury Apartment': 37, 'Blueground | Amueblado, Security & Business Center': 33, 'Casa Miravalle | Condesa': 21}
   - host_name: 4300 valores únicos
      Principales valores:
{'Alejandro': 388, 'Alejandra': 322, 'Raul': 274, 'Luis': 269, 'Francisco': 235}
   - neighbourhood: 16 valores únicos
      Principales valores:
{'Cuauhtémoc': 12514, 'Miguel Hidalgo': 4573, 'Benito Juárez': 3087, 'Coyoacán': 1727, 'Álvaro Obregón': 978}
   - room_type: 4 valores únicos
      Principales valores:
{'Entire home/apt': 17713, 'Private room': 8995, 'Shared room': 257, 'Hotel room': 86}
   

### Análisis de valores nulos

In [5]:
analizar_valores_nulos(df_listings, "LISTINGS")


VALORES NULOS Y FALTANTES - LISTINGS:
--------------------------------------------------------------------------------
   Columnas con valores faltantes:


,Columna,Cantidad de nulos,Porcentaje (%)
9,price,3484,12.88
15,last_review,3401,12.57
16,reviews_per_month,3401,12.57
4,host_name,6,0.02


### Análisis de duplicados

In [6]:
analizar_duplicados(df_listings, "LISTINGS", col_id='id')


REGISTROS DUPLICADOS - LISTINGS:
--------------------------------------------------------------------------------
   1. POR COLUMNA ID (id):
      - Total de registros: 27,051
      - Registros únicos: 27,051
      - Duplicados por ID: 0 (0.00%)

   2. FILAS COMPLETAMENTE DUPLICADAS (todas las columnas):
      - Filas duplicadas: 0 (0.00%)

   Excelente: No hay duplicados en esta colección


### Análisis de outliers

In [7]:
analizar_outliers(df_listings, "LISTINGS", columnas=["price", "minimum_nights", "availability_365"])


ANÁLISIS DE OUTLIERS - LISTINGS:
--------------------------------------------------------------------------------

   price:
      - Mínimo: 61.00
      - Q1 (25%): 643.00
      - Mediana: 1039.00
      - Q3 (75%): 1611.00
      - Máximo: 900000.00
      - IQR: 968.00
      - Outliers detectados: 1,768 (7.50%)
      - Rango válido: [-809.00, 3063.00]

   minimum_nights:
      - Mínimo: 1.00
      - Q1 (25%): 1.00
      - Mediana: 2.00
      - Q3 (75%): 2.00
      - Máximo: 1125.00
      - IQR: 1.00
      - Outliers detectados: 3,429 (12.68%)
      - Rango válido: [-0.50, 3.50]

   availability_365:
      - Mínimo: 0.00
      - Q1 (25%): 140.00
      - Mediana: 269.00
      - Q3 (75%): 341.00
      - Máximo: 365.00
      - IQR: 201.00
      - Outliers detectados: 0 (0.00%)
      - Rango válido: [-161.50, 642.50]


## 2. Análisis de Colección: REVIEWS

Información sobre las reseñas que dejan los usuarios de los apartamentos.

### Análisis de estructura

In [8]:
analizar_estructura(df_reviews, "REVIEWS")


COLECCIÓN: REVIEWS

PRIMERAS FILAS (primeros 5 registros):


,_id,listing_id,date
0,69dbd860dcaeaf06fe060c7e,44616,2011-11-09
1,69dbd860dcaeaf06fe060c7f,44616,2012-08-16
2,69dbd860dcaeaf06fe060c80,44616,2012-12-28
3,69dbd860dcaeaf06fe060c81,44616,2013-01-04
4,69dbd860dcaeaf06fe060c82,44616,2013-03-19



DIMENSIONES:
   - Filas: 1,454,740
   - Columnas: 3

TIPOS DE DATOS:
INFO
<class 'pandas.DataFrame'>
RangeIndex: 1454740 entries, 0 to 1454739
Columns: 3 entries, _id to date
dtypes: datetime64[us](1), int64(1), str(1)
memory usage: 33.3 MB
TYPES
_id                      str
listing_id             int64
date          datetime64[us]
dtype: object

ESTADÍSTICAS (Columnas numéricas):


,count,mean,min,25%,50%,75%,max,std
listing_id,1454740.0,355821508552638016.0,44616.0,26850859.0,46919861.0,791170307741018368.0,1517372885452894720.0,471299866764997760.0
date,1454740,2023-02-18 02:05:34.001128,2011-04-02 00:00:00,2022-02-02 00:00:00,2023-10-17 00:00:00,2024-11-18 00:00:00,2025-09-28 00:00:00,NaN



ESTADÍSTICAS COMPLETAS (incluye variables categóricas):


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
_id,1454740,1454740,69dbd860dcaeaf06fe060c7e,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
listing_id,1454740.0,NaN,NaN,NaN,355821508552638016.0,44616.0,26850859.0,46919861.0,791170307741018368.0,1517372885452894720.0,471299866764997760.0
date,1454740,NaN,NaN,NaN,2023-02-18 02:05:34.001128,2011-04-02 00:00:00,2022-02-02 00:00:00,2023-10-17 00:00:00,2024-11-18 00:00:00,2025-09-28 00:00:00,NaN



COLUMNAS NO NUMÉRICAS (2):
   - _id: 1454740 valores únicos
      Principales valores:
{'69dbd860dcaeaf06fe060c7e': 1, '69dbd860dcaeaf06fe060c7f': 1, '69dbd860dcaeaf06fe060c80': 1, '69dbd860dcaeaf06fe060c81': 1, '69dbd860dcaeaf06fe060c82': 1}
   - date: 4754 valores únicos
      Principales valores:
{Timestamp('2025-08-31 00:00:00'): 2753, Timestamp('2025-06-29 00:00:00'): 2657, Timestamp('2025-03-17 00:00:00'): 2542, Timestamp('2025-04-27 00:00:00'): 2520, Timestamp('2025-02-24 00:00:00'): 2510}


### Análisis de valores nulos

In [9]:
analizar_valores_nulos(df_reviews, "REVIEWS")


VALORES NULOS Y FALTANTES - REVIEWS:
--------------------------------------------------------------------------------
   Todas las columnas están completas (sin nulos)


### Análisis de duplicados

In [10]:
analizar_duplicados(df_reviews, "REVIEWS", col_id='_id')


REGISTROS DUPLICADOS - REVIEWS:
--------------------------------------------------------------------------------
   1. POR COLUMNA ID (_id):
      - Total de registros: 1,454,740
      - Registros únicos: 1,454,740
      - Duplicados por ID: 0 (0.00%)

   2. FILAS COMPLETAMENTE DUPLICADAS (todas las columnas):
      - Filas duplicadas: 0 (0.00%)

   Excelente: No hay duplicados en esta colección


### Análisis de outliers

In [11]:
analizar_outliers(df_reviews, "REVIEWS")


ANÁLISIS DE OUTLIERS - REVIEWS:
--------------------------------------------------------------------------------

   listing_id:
      - Mínimo: 44616.00
      - Q1 (25%): 26850859.00
      - Mediana: 46919861.00
      - Q3 (75%): 791170307741018368.00
      - Máximo: 1517372885452894720.00
      - IQR: 791170307714167552.00
      - Outliers detectados: 0 (0.00%)
      - Rango válido: [-1186755461544400384.00, 1977925769312269568.00]


## 3. Análisis de Colección: CALENDAR

Información sobre disponibilidad, precios y políticas de noches mínimas para cada día.

### Análisis de estructura

In [12]:
analizar_estructura(df_calendar, "CALENDAR")


COLECCIÓN: CALENDAR

PRIMERAS FILAS (primeros 5 registros):


,_id,listing_id,date,available,minimum_nights,maximum_nights
0,69dbd65cdcaeaf06fe6f63a4,7860479,2025-09-28,False,5,28
1,69dbd65cdcaeaf06fe6f63a5,7860479,2025-09-29,False,5,28
2,69dbd65cdcaeaf06fe6f63a6,7860479,2025-09-30,False,5,28
3,69dbd65cdcaeaf06fe6f63a7,7860479,2025-10-01,False,5,28
4,69dbd65cdcaeaf06fe6f63a8,7860479,2025-10-02,False,5,28



DIMENSIONES:
   - Filas: 9,873,624
   - Columnas: 6

TIPOS DE DATOS:
INFO
<class 'pandas.DataFrame'>
RangeIndex: 9873624 entries, 0 to 9873623
Columns: 6 entries, _id to maximum_nights
dtypes: bool(1), datetime64[us](1), int64(3), str(1)
memory usage: 386.1 MB
TYPES
_id                          str
listing_id                 int64
date              datetime64[us]
available                   bool
minimum_nights             int64
maximum_nights             int64
dtype: object

ESTADÍSTICAS (Columnas numéricas):


,count,mean,min,25%,50%,75%,max,std
listing_id,9873624.0,700355870257449216.0,35797.0,44104681.0,830736761677326976.0,1217411034178283008.0,1518561444972990720.0,570260905803648384.0
date,9873624,2026-03-28 17:33:26.990745,2025-09-27 00:00:00,2025-12-27 00:00:00,2026-03-29 00:00:00,2026-06-28 00:00:00,2026-09-27 00:00:00,NaN
minimum_nights,9873624.0,4.309379,1.0,1.0,2.0,2.0,1125.0,23.623162
maximum_nights,9873624.0,794562.366399,1.0,365.0,730.0,1125.0,2147483647.0,41281702.330175



ESTADÍSTICAS COMPLETAS (incluye variables categóricas):


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
_id,9873624,9873624,69dbd65cdcaeaf06fe6f63a4,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
listing_id,9873624.0,NaN,NaN,NaN,700355870257449216.0,35797.0,44104681.0,830736761677326976.0,1217411034178283008.0,1518561444972990720.0,570260905803648384.0
date,9873624,NaN,NaN,NaN,2026-03-28 17:33:26.990745,2025-09-27 00:00:00,2025-12-27 00:00:00,2026-03-29 00:00:00,2026-06-28 00:00:00,2026-09-27 00:00:00,NaN
available,9873624,2,True,6299554,NaN,NaN,NaN,NaN,NaN,NaN,NaN
minimum_nights,9873624.0,NaN,NaN,NaN,4.309379,1.0,1.0,2.0,2.0,1125.0,23.623162
maximum_nights,9873624.0,NaN,NaN,NaN,794562.366399,1.0,365.0,730.0,1125.0,2147483647.0,41281702.330175



COLUMNAS NO NUMÉRICAS (3):
   - _id: 9873624 valores únicos
      Principales valores:
{'69dbd65cdcaeaf06fe6f63a4': 1, '69dbd65cdcaeaf06fe6f63a5': 1, '69dbd65cdcaeaf06fe6f63a6': 1, '69dbd65cdcaeaf06fe6f63a7': 1, '69dbd65cdcaeaf06fe6f63a8': 1}
   - date: 366 valores únicos
      Principales valores:
{Timestamp('2025-09-28 00:00:00'): 27051, Timestamp('2025-09-29 00:00:00'): 27051, Timestamp('2025-09-30 00:00:00'): 27051, Timestamp('2025-10-01 00:00:00'): 27051, Timestamp('2025-10-02 00:00:00'): 27051}
   - available: 2 valores únicos
      Principales valores:
{True: 6299554, False: 3574070}


### Análisis de valores nulos

In [13]:
analizar_valores_nulos(df_calendar, "CALENDAR")


VALORES NULOS Y FALTANTES - CALENDAR:
--------------------------------------------------------------------------------
   Todas las columnas están completas (sin nulos)


### Análisis de duplicados

In [14]:
analizar_duplicados(df_calendar, "CALENDAR", col_id='_id')


REGISTROS DUPLICADOS - CALENDAR:
--------------------------------------------------------------------------------
   1. POR COLUMNA ID (_id):
      - Total de registros: 9,873,624
      - Registros únicos: 9,873,624
      - Duplicados por ID: 0 (0.00%)

   2. FILAS COMPLETAMENTE DUPLICADAS (todas las columnas):
      - Filas duplicadas: 0 (0.00%)

   Excelente: No hay duplicados en esta colección


### Análisis de outliers

In [15]:
analizar_outliers(df_calendar, "CALENDAR", columnas=["maximum_nights", "minimum_nights"])


ANÁLISIS DE OUTLIERS - CALENDAR:
--------------------------------------------------------------------------------

   maximum_nights:
      - Mínimo: 1.00
      - Q1 (25%): 365.00
      - Mediana: 730.00
      - Q3 (75%): 1125.00
      - Máximo: 2147483647.00
      - IQR: 760.00
      - Outliers detectados: 7,280 (0.07%)
      - Rango válido: [-775.00, 2265.00]

   minimum_nights:
      - Mínimo: 1.00
      - Q1 (25%): 1.00
      - Mediana: 2.00
      - Q3 (75%): 2.00
      - Máximo: 1125.00
      - IQR: 1.00
      - Outliers detectados: 1,296,995 (13.14%)
      - Rango válido: [-0.50, 3.50]
